# 开发错误助手（Dev Error Helper）

## 练习目标（理念）

用大模型把冗长的 **Git / GitHub / Python / 终端** 报错，翻译成学生能听懂的话，并给出**安全的下一步**（尤其要标出可能删代码、覆盖他人工作的危险命令）。

这是第 1 周 Chat Completions 的实战小品：`system` 定「怎么解释」，`user` 贴真实报错，一次 `create` 拿到结构化 Markdown 回答。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages`（system / user） | `system_prompt` 规定解释格式；`user_prompt` 放命令 + 报错全文 |
| Chat Completions | `openai.chat.completions.create(model=..., messages=...)` |
| 环境变量与密钥 | `load_dotenv` + `OpenAI()` 读 `OPENAI_API_KEY` |

## 怎么跑

1. 准备 `.env`：设置好 `OPENAI_API_KEY`
2. 依次运行下面两个代码格
3. 想换场景时：只改 `user_prompt` 里的命令与报错文本，再重跑调用格



In [ ]:
# ========== 导入与环境：准备好 OpenAI 客户端 ==========

# 导入标准库 os：需要时可读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# override=True：.env 中的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 创建客户端；未显式传 api_key 时，SDK 默认读环境变量 OPENAI_API_KEY
openai = OpenAI()




In [ ]:
# ========== 开发错误助手：解释 Git 报错并建议安全下一步 ==========


# ----- 第 1 步：编写提示词（发给模型的英文保留原样，改译会改变回答风格） -----

# system_prompt：规定导师角色 + 固定 Markdown 小节结构（What / Important / Why / Next / Warning）
system_prompt = """ 

        You are a helpful coding mentor for students who are learning Git, GitHub, Python, and terminal commands.

        When I give you a long error message, explain it in a simple way.

        Use this format:

        ## What the error means
        Explain the error in simple words so students/beginners can understand easily

        ## Important part of the error
        Show the main line or phrase that explains the problem.

        ## Why it probably happened
        Explain the likely reason.

        ## What I should do next
        Give safe steps with commands and explanations what each does ,keep it short

        ## Warning
        Warn me if there is any command that could delete work or overwrite someone else's work.

        Keep it short, simple, and easy for a student to understand.
        Do not over-explain.
        Do not give too many options.

"""

# user_prompt：真实场景——本地 feature 分支 push 被 non-fast-forward 拒绝（字符串内容不翻译）
user_prompt = """
    I am a junior developer and got a Git error message

    command I ran : git push origin feature/append

    error message:To https://github.com/company/qc-pipeline.git
 ! [rejected]        feature/append -> feature/append (non-fast-forward)
error: failed to push some refs to 'https://github.com/company/qc-pipeline.git'
hint: Updates were rejected because the tip of your current branch is behind
hint: its remote counterpart. Integrate the remote changes before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.

Context:
I was working on a feature branch. I committed my changes locally, but when I tried to push, Git rejected it.
"""

# ----- 第 2 步：组装 messages 列表（system 定规则，user 放具体报错） -----

messages = [{"role":"system","content":system_prompt}, {"role":"user","content":user_prompt}] # 填入系统与用户消息

# ----- 第 3 步：调用 OpenAI Chat Completions（model id 保留原样） -----
response =openai.chat.completions.create(model="gpt-5-nano", messages=messages)

# ----- 第 4 步：打印助手回复正文 -----
print(response.choices[0].message.content)


